# 🛡️ Recon Framework

## Complete Machine Learning Pipeline

This notebook implements an end-to-end ML pipeline for detecting and classifying cyber threats in network traffic.

**Pipeline Steps:**
1. Dataset Loading (User-Provided)
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Model Training (Random Forest Classifier)
5. Model Evaluation
6. Model Serialization (Pickle Files)

---

## 📦 Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import pickle
import json

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12



ModuleNotFoundError: No module named 'pandas'

## 📂 Step 2: Load Your Dataset

**Please provide the path to your dataset CSV file when prompted.**

The dataset should contain network traffic features with a target/label column for attack classification.

In [ ]:

dataset_path = input("📁 Enter the full path to your dataset CSV file: ").strip()

dataset_path = dataset_path.strip('"').strip("'")

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ File not found: {dataset_path}")

if dataset_path.lower().endswith('.arff'):
    column_names = []
    data_lines = []
    in_data = False
    with open(dataset_path, 'r') as f_in:
        for file_line in f_in:
            file_line = file_line.strip()
            if not file_line or file_line.startswith('%'):
                continue
            if file_line.lower() == '@data':
                in_data = True
                continue
            if in_data:
                data_lines.append(file_line)
            elif file_line.lower().startswith('@attribute'):
                parts = file_line.split("'")
                if len(parts) >= 2:
                    col_name = parts[1]
                else:
                    col_name = file_line.split()[1]
                column_names.append(col_name)
    from io import StringIO
    csv_text = '\n'.join(data_lines)
    df = pd.read_csv(StringIO(csv_text), header=None, names=column_names)
else:
    df = pd.read_csv(dataset_path)

print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
for i, col in enumerate(df.columns, 1):
    print(f"  {i:3d}. {col}")

print(f"\n📊 Total Columns: {len(df.columns)}")

In [ ]:
target_column = input("🎯 Enter the name of the TARGET/LABEL column (e.g., 'label', 'attack_type', 'class'): ").strip()

if target_column not in df.columns:
    raise ValueError(f"❌ Column '{target_column}' not found in dataset. Available columns: {list(df.columns)}")

print(df[target_column].value_counts())

## 📊 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print(f"Shape: {df.shape}")
print(f"Data Types:")
print(df.dtypes.value_counts())
print(f"Missing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values found!")
else:
    print(missing[missing > 0])
print(f"Statistical Summary:")
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#00d2ff', '#ff6b6b', '#feca57', '#48dbfb', '#ff9ff3']
class_counts = df[target_column].value_counts()
axes[0].bar(class_counts.index.astype(str), class_counts.values, color=colors[:len(class_counts)], edgecolor='white', linewidth=0.5)
axes[0].set_title('🎯 Target Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(class_counts.values, labels=class_counts.index.astype(str), autopct='%1.1f%%',
            colors=colors[:len(class_counts)], shadow=True, startangle=140)
axes[1].set_title('📊 Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numerical_cols) > 1:
    top_features = df[numerical_cols].var().nlargest(min(20, len(numerical_cols))).index.tolist()
    
    plt.figure(figsize=(14, 10))
    correlation_matrix = df[top_features].corr()
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm',
                center=0, square=True, linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
    plt.title('🔥 Feature Correlation Heatmap (Top Features by Variance)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("⚠️ Not enough numerical columns for correlation heatmap")

In [ ]:
if len(numerical_cols) >= 4:
    top_4 = df[numerical_cols].var().nlargest(4).index.tolist()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    for idx, col in enumerate(top_4):
        ax = axes[idx // 2][idx % 2]
        ax.hist(df[col], bins=50, color=colors[idx], alpha=0.7, edgecolor='white')
        ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
    
    plt.suptitle('📊 Top Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("⚠️ Not enough numerical features for distribution plots")

## ⚙️ Step 4: Data Preprocessing

In [ ]:
data = df.copy()

missing_before = data.isnull().sum().sum()
if missing_before > 0:
    num_cols = data.select_dtypes(include=[np.number]).columns
    data[num_cols] = data[num_cols].fillna(data[num_cols].median())
    
    cat_cols = data.select_dtypes(include=['object']).columns
    for col in cat_cols:
        data[col] = data[col].fillna(data[col].mode()[0])
    
    print(f"Filled {missing_before} missing values")
else:
    print("No missing values to handle")

dup_count = data.duplicated().sum()
if dup_count > 0:
    data = data.drop_duplicates()
    print(f"Removed {dup_count} duplicate rows")
else:
    print("No duplicate rows found")

print(f"\n📊 Processed dataset shape: {data.shape}")

In [ ]:

categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
if target_column in categorical_cols:
    categorical_cols.remove(target_column)

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded '{col}': {len(le.classes_)} unique values")

target_encoder = LabelEncoder()
data[target_column] = target_encoder.fit_transform(data[target_column].astype(str))
print(f"Target classes: {list(target_encoder.classes_)}")
print(f"   Encoded as: {list(range(len(target_encoder.classes_)))}")

In [ ]:
X = data.drop(columns=[target_column])
y = data[target_column]

feature_names = X.columns.tolist()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\n📋 Feature Names ({len(feature_names)} total):")


In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_names)

print(f"StandardScaler applied to {X_scaled.shape[1]} features")
print(f"Mean (should be ~0): {X_scaled.mean().mean():.6f}")
print(f"Std  (should be ~1): {X_scaled.std().mean():.6f}")

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"Stratified split applied to maintain class balance")

## 🤖 Step 5: Model Training

In [ ]:

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

model.fit(X_train, y_train)

print(f"\nModel trained successfully!")
print(f"Trees: {model.n_estimators}")
print(f"Features: {model.n_features_in_}")
print(f"Classes: {model.n_classes_}")

## 📈 Step 6: Model Evaluation

In [ ]:

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_encoder.classes_,
            yticklabels=target_encoder.classes_,
            linewidths=0.5, linecolor='white')
plt.title('🎯 Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

top_n = min(15, len(feature_importance))
plt.figure(figsize=(12, 6))
top_features_df = feature_importance.head(top_n)
plt.barh(range(top_n), top_features_df['Importance'].values, color='#00d2ff', edgecolor='white')
plt.yticks(range(top_n), top_features_df['Feature'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.title(f'🏆 Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Top 10 Features:")
print(feature_importance.head(10).to_string(index=False))

## 💾 Step 7: Save Model Artifacts (Pickle Files)

Saving the trained model, scaler, label encoders, and metadata as pickle files for deployment.

In [ ]:
models_dir = os.path.join(os.path.dirname(os.getcwd()), 'models')
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, 'model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved: {model_path}")

scaler_path = os.path.join(models_dir, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"Scaler saved: {scaler_path}")

encoder_path = os.path.join(models_dir, 'label_encoder.pkl')
with open(encoder_path, 'wb') as f:
    pickle.dump(target_encoder, f)
print(f"Label Encoder saved: {encoder_path}")

feature_encoders_path = os.path.join(models_dir, 'feature_encoders.pkl')
with open(feature_encoders_path, 'wb') as f:
    pickle.dump(label_encoders, f)
print(f"Feature Encoders saved: {feature_encoders_path}")

metadata = {
    'feature_names': feature_names,
    'target_classes': list(target_encoder.classes_),
    'categorical_columns': categorical_cols,
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'n_estimators': model.n_estimators,
    'n_features': len(feature_names),
    'n_classes': len(target_encoder.classes_),
    'training_samples': int(X_train.shape[0]),
    'testing_samples': int(X_test.shape[0])
}

metadata_path = os.path.join(models_dir, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"Metadata saved: {metadata_path}")

for f_name in os.listdir(models_dir):
    f_path = os.path.join(models_dir, f_name)
    f_size = os.path.getsize(f_path) / 1024
    print(f"  📄 {f_name} ({f_size:.1f} KB)")

In [ ]:
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")
